<a href="https://colab.research.google.com/github/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/starter/starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Research Foundations Multilingual Tokenization Challenge

## Can you build the most efficient multilingual tokenizer?

You have six languages, one vocabulary, and a budget of 10,000 tokens.

The full overview, dataset description, rules, scoring formula and submission
steps are in the [competition README](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge).

---

## What This Notebook Provides

This notebook gives you a starting point for the competition:

* Code to load and explore the **training and validation data**.
* A **standard BPE baseline** that defines the reference score.
* A **balanced BPE baseline** showing how one change to the data strategy affects multilingual tokenization.
* The official **validation scoring function**.
* Per-language results for comparing tokenizer performance.
* A **submission checker** for validating your final tokenizer.
* Code to export your tokenizer as `tokenizer.json`.

The baselines are starting points, not solutions.

# Getting Started

Install the pinned competition runtime and download the submission checker.
`tokenizers` must be exactly `0.22.1`, because that is the version official
evaluation uses.

In [ ]:
# Colab and Kaggle need the pinned competition runtime installed. A local
# environment created with `uv sync --dev` already has it, and uv virtual
# environments do not ship pip, so this cell installs only what is missing.
from importlib.metadata import PackageNotFoundError, version

TOKENIZERS_VERSION = "0.22.1"


def needs_install(package, exact=None):
    """Return True when a package is absent or not at the required version.

    Args:
        package: Distribution name to look up.
        exact: Version that must match exactly, or None for any version.

    Returns:
        True when pip should install the package.
    """
    try:
        found = version(package)
    except PackageNotFoundError:
        return True
    return exact is not None and found != exact


missing = []
if needs_install("tokenizers", TOKENIZERS_VERSION):
    missing.append(f"tokenizers=={TOKENIZERS_VERSION}")
for package, requirement in (("datasets", "datasets>=4.0,<5"),
                             ("pandas", "pandas"),
                             ("matplotlib", "matplotlib")):
    if needs_install(package):
        missing.append(requirement)

if missing:
    packages = " ".join(missing)
    !pip install -q {packages}
else:
    print("Dependencies already available at the required versions.")

In [ ]:
# Competition configuration.
GITHUB_REPO = "aims-ai-research-foundations/airf-multilingual-tokenizer-challenge"
GITHUB_BRANCH = "main"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {
    "en": "English",
    "fr": "French",
    "ha": "Hausa",
    "sw": "Swahili",
    "yo": "Yoruba",
    "am": "Amharic",
}
MAX_VOCAB_SIZE = 10_000 # Don't change this. The evaluation script will fail if you do.

# Only these four languages are scored. English and French are guardrails.
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
GUARDRAIL_RATIO = 1.15
UNKNOWN_PENALTY = 100
RECONSTRUCTION_PENALTY = 3

RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}"

In [ ]:
# Make the submission checker importable. A local clone already ships it in
# `starter/`, and Colab or Kaggle downloads it once. This uses urllib rather
# than a shell tool, so it behaves the same on every platform.
import sys
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tokenizers
from datasets import load_dataset
from tokenizers import (Tokenizer, decoders, models, normalizers,
                        pre_tokenizers, trainers)


def local_file(relative_path):
    """Find a repository file when the notebook runs inside a clone.

    Args:
        relative_path: Path relative to the repository root.

    Returns:
        An existing Path, or None when the notebook is not inside a clone.
    """
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    return None


def ensure_utils():
    """Put utils.py on the import path, downloading it only when needed.

    Returns:
        The directory added to `sys.path`.
    """
    found = local_file("starter/utils.py") or local_file("utils.py")
    if found is None:
        urllib.request.urlretrieve(f"{RAW_URL}/starter/utils.py", "utils.py")
        found = Path("utils.py").resolve()
    directory = str(found.parent)
    if directory not in sys.path:
        sys.path.insert(0, directory)
    return directory


ensure_utils()
from utils import profile_submission

print("tokenizers version:", tokenizers.__version__)

## Load the Dataset

Let's start by loading the competition data and taking a look at what we're working with.

Before building a tokenizer, inspect the size of the dataset, the distribution of languages, and a few examples from each language.

Understanding your data is part of understanding your tokenizer.

In [ ]:
def load_competition_data(split):
    """Load one public competition split as a pandas DataFrame.

    Args:
        split: Either "train" or "validation".

    Returns:
        A DataFrame with a `language` column and a `text` column.
    """
    if split not in {"train", "validation"}:
        raise ValueError("only the train and validation splits are public")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION)
    frame = dataset.to_pandas()[["language", "text"]]
    return frame.reset_index(drop=True)


train = load_competition_data("train")
validation = load_competition_data("validation")

print(f"Train rows:      {len(train):,}")
print(f"Validation rows: {len(validation):,}")

In [ ]:
# Rows are balanced across languages, but characters are not.
summary = train.assign(characters=train.text.str.len()).groupby("language").agg(
    rows=("text", "size"),
    characters=("characters", "sum"),
    mean_length=("characters", "mean"),
)

summary["share_of_characters"] = summary.characters / summary.characters.sum()
summary.index = [LANGUAGE_NAMES[language] for language in summary.index]
summary.sort_values("characters", ascending=False).round(
    {"mean_length": 1, "share_of_characters": 3}
)

In [ ]:
# Every language contributes the same number of rows but not the same amount of
# text, and the tokenizer learns its vocabulary from characters, not from rows.
sizes = summary.characters.sort_values() / 1e6

figure, axes = plt.subplots(figsize=(7.2, 3.4))
bars = axes.barh(sizes.index, sizes.values, height=0.62, color="#4878a8")
axes.bar_label(bars, fmt="%.2fM", padding=5, fontsize=9, color="#444444")

axes.set_xlabel("Characters in the training split (millions)")
axes.set_title("Same rows per language, different amounts of text", pad=12)
axes.set_xlim(0, sizes.max() * 1.18)
axes.xaxis.grid(True, color="#e8e8e8", linewidth=0.8)
axes.set_axisbelow(True)
axes.tick_params(axis="y", length=0)
for edge in ("top", "right", "left"):
    axes.spines[edge].set_visible(False)

plt.tight_layout()
plt.show()

spread = summary.characters.max() / summary.characters.min()
print(f"Largest language carries {spread:.2f}x the characters of the smallest.")

In [ ]:
# One random example per language. Re-run this cell to see different text.
for language in LANGUAGES:
    text = train.loc[train.language == language, "text"].sample(1).iloc[0]
    print(f"[{language}] {LANGUAGE_NAMES[language]}")
    print(f"     {text[:120]}")

# Baseline 1: Word Level

## The Efficient Extreme

The simplest way to use few tokens is to give every word its own token.

With a 10,000 token vocabulary you can cover the most frequent words in the
corpus, and each of those becomes exactly one token, so $F_l$ approaches the
theoretical floor of $1.0$.

The problem is everything else. Any word outside the vocabulary becomes
`[UNK]`, and each 1 percent of words that fall back adds 1.00 to the score.
Watch what that does to a tokenizer whose raw fertility is the best possible.

In [ ]:
def train_word_level(texts, vocab_size=MAX_VOCAB_SIZE):
    """Give each frequent word a single token, with [UNK] for the rest.

    Args:
        texts: An iterable of training strings.
        vocab_size: The vocabulary budget, at most 10,000.

    Returns:
        A trained `tokenizers.Tokenizer`.
    """
    tokenizer = Tokenizer(models.WordLevel(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.WordLevelTrainer(
        vocab_size=vocab_size, special_tokens=["[UNK]"], show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer, length=len(texts))
    return tokenizer


word_level = train_word_level(train.text.tolist())
print(f"Learned vocabulary: "
      f"{word_level.get_vocab_size(with_added_tokens=True):,} / {MAX_VOCAB_SIZE:,}")

## Score the Baselines

For each language $l$, token fertility is tokens divided by words:

$$F_l = \frac{\text{Number of tokens produced}}{\text{Number of words}}$$

On its own that would reward a tokenizer for refusing to represent hard words,
so unknown tokens carry a heavy penalty:

$$U_l = \frac{\text{Number of [UNK] tokens}}{\text{Number of words}}
\qquad
S_l = F_l + 100\,U_l$$

A 1 percent unknown rate adds $1.00$ to that language's score. Your final score
is the average of $S_l$ across the four target languages:

$$\text{Score} = \frac{S_{\text{ha}} + S_{\text{sw}} + S_{\text{yo}} + S_{\text{am}}}{4}$$

**Lower is better.**

English and French are not scored directly. Each may reach $1.15$ times your
four-language average at no cost, and anything above that is added to your
score:

$$P = \sum_{l \in \{\text{en}, \text{fr}\}} \max\left(0,\ F_l - 1.15\,\bar{F}\right)
\qquad
\text{Score} = \frac{S_{\text{ha}} + S_{\text{sw}} + S_{\text{yo}} + S_{\text{am}}}{4} + P$$

where $\bar{F}$ is your average fertility across the four target languages. Stay
inside the limit and $P$ is exactly zero. Without this the best move is to
abandon English and French, which is not a multilingual tokenizer.

One more term, for the same reason. **Reconstruction** is the share of rows
where `decode(encode(text))` exactly matches the original, and any failure is
charged at three times that share:

$$P_{\text{reconstruction}} = 3 \times (1 - \text{reconstruction})$$

This discourages approaches that artificially reduce token counts by modifying
or deleting information from the input. Unicode NFC, special tokens and a
leading space are all ignored, so a tokenizer that reconstructs every example
receives no penalty.

In [ ]:
def count_words(text):
    """Count whitespace-separated words in the original text."""
    return len(text.split())


def measure(tokenizer, data):
    """Compute tokens per word, unknown tokens per word, and reconstruction.

    Args:
        tokenizer: A trained `tokenizers.Tokenizer`.
        data: A DataFrame with `language` and `text` columns.

    Returns:
        A tuple of two dictionaries, fertility and unknown rate, plus the
        share of rows the tokenizer reconstructs exactly.
    """
    tokens = defaultdict(int)
    words = defaultdict(int)
    unknowns = defaultdict(int)
    rebuilt = 0
    unknown_id = tokenizer.token_to_id("[UNK]")
    texts = data.text.tolist()
    encodings = tokenizer.encode_batch(texts, add_special_tokens=False)
    for language, text, encoding in zip(data.language, texts, encodings):
        tokens[language] += len(encoding.ids)
        words[language] += count_words(text)
        if unknown_id is not None:
            unknowns[language] += sum(1 for i in encoding.ids if i == unknown_id)
        # NFC, special tokens and outer whitespace are ignored, because none of
        # them discards any of the original text.
        restored = tokenizer.decode(encoding.ids, skip_special_tokens=True)
        rebuilt += (unicodedata.normalize("NFC", restored).strip()
                    == unicodedata.normalize("NFC", text).strip())
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES}
    unknown_rate = {l: unknowns[l] / words[l] for l in LANGUAGES}
    return fertility, unknown_rate, rebuilt / len(texts)


def score(tokenizer, data=validation, name="candidate"):
    """Score a tokenizer with the official competition metric.

    Returns:
        A dictionary with the score, per language detail, and the guardrail and
        reconstruction penalties that are included in that score.
    """
    fertility, unknown_rate, reconstruction = measure(tokenizer, data)
    penalised = {l: fertility[l] + UNKNOWN_PENALTY * unknown_rate[l] for l in LANGUAGES}
    value = sum(penalised[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = (sum(fertility[l] for l in SCORED_LANGUAGES)
              / len(SCORED_LANGUAGES)) * GUARDRAIL_RATIO
    overages = {l: max(0.0, fertility[l] - budget) for l in ("en", "fr")}
    penalty = sum(overages.values())
    reconstruction_penalty = RECONSTRUCTION_PENALTY * (1 - reconstruction)
    return {
        "name": name,
        "score": value + penalty + reconstruction_penalty,
        "fertility": fertility,
        "unknown_rate": unknown_rate,
        "penalised": penalised,
        "budget": budget,
        "penalty": penalty,
        "overages": overages,
        "reconstruction": reconstruction,
        "reconstruction_penalty": reconstruction_penalty,
        "vocab_size": tokenizer.get_vocab_size(with_added_tokens=True),
    }


def show_score(result):
    """Print one scored result as a per language table."""
    print(f"{result['name']}: SCORE {result['score']:.4f}   "
          f"vocabulary {result['vocab_size']:,}")
    print(f"  {'language':<10}{'tokens/word':>13}{'[UNK] rate':>12}{'score':>9}")
    for language in LANGUAGES:
        marker = "*" if language in SCORED_LANGUAGES else " "
        print(f"  {LANGUAGE_NAMES[language] + marker:<10}"
              f"{result['fertility'][language]:>13.3f}"
              f"{result['unknown_rate'][language]:>12.4f}"
              f"{result['penalised'][language]:>9.3f}")
    print(f"  guardrail {result['budget']:.3f}, penalty {result['penalty']:.4f}")
    for language, overage in result["overages"].items():
        if overage > 0:
            print(f"  {LANGUAGE_NAMES[language]} is {overage:.3f} over, "
                  f"adding {overage:.3f} to the score")
    print(f"  reconstruction {result['reconstruction'] * 100:.1f}%, "
          f"penalty {result['reconstruction_penalty']:.4f}")


word_result = score(word_level, name="Baseline 1: Word level")
show_score(word_result)

# Baseline 2: Character Level

## The Complete Extreme

Now go the other way. Give every character its own token, and fall back to raw
bytes for anything unseen.

This tokenizer can represent absolutely everything, so its `[UNK]` rate is
zero and it pays no penalty at all. It is also enormously wasteful: a word of
eight characters costs eight tokens.

**Same vocabulary budget. Opposite trade-off.**

In [ ]:
def train_character_level(texts, min_count=20):
    """Give each common character a single token, with byte fallback.

    Byte fallback is what satisfies the coverage requirement: a character
    missing from the vocabulary is still representable as raw bytes.

    Args:
        texts: An iterable of training strings.
        min_count: How often a character must appear to earn its own token.

    Returns:
        A `tokenizers.Tokenizer` that represents any input.
    """
    counts = Counter(character for text in texts for character in text)
    alphabet = [character for character, seen in counts.most_common()
                if seen >= min_count]

    vocab = {"[UNK]": 0}
    for character in alphabet:
        vocab.setdefault(character, len(vocab))
    for value in range(256):
        vocab.setdefault(f"<0x{value:02X}>", len(vocab))

    model = models.BPE(vocab=vocab, merges=[], unk_token="[UNK]",
                       byte_fallback=True)
    tokenizer = Tokenizer(model)
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.decoder = decoders.ByteFallback()
    return tokenizer


character_level = train_character_level(train.text.tolist())
size = character_level.get_vocab_size(with_added_tokens=True)
print(f"Vocabulary: {size:,} / {MAX_VOCAB_SIZE:,}")

character_result = score(character_level, name="Baseline 2: Character level")
show_score(character_result)

## Compare the Baselines

The two baselines bracket the problem.

| Tokenizer | Tokens per word | `[UNK]` rate | Score |
| --- | ---: | ---: | ---: |
| Word level | lowest possible | high | terrible |
| Character level | high | zero | mediocre |

Neither is a good tokenizer. One wins on fertility and loses everything to the
penalty, the other pays no penalty but spends far too many tokens.

**Your job is to find something in between.** A good subword tokenizer keeps
full coverage while spending far fewer tokens per word than character level.

In [ ]:
# Both baselines side by side, with the per language detail.
comparison = pd.DataFrame(
    [
        {"tokenizer": result["name"],
         "score": round(result["score"], 4),
         "fertility": round(
             sum(result["fertility"][l] for l in SCORED_LANGUAGES) / 4, 3),
         "unk rate": round(
             sum(result["unknown_rate"][l] for l in SCORED_LANGUAGES) / 4, 4),
         "vocabulary": result["vocab_size"],
         **{language: round(result["penalised"][language], 3)
            for language in LANGUAGES}}
        for result in (word_result, character_result)
    ]
).set_index("tokenizer")

comparison

# Your Turn

## Can You Beat the Baselines?

You have seen the two extremes. One is efficient but cannot represent unseen
words. The other represents everything but spends far too many tokens.

A subword tokenizer sits between them, and that is where the competition is.

Here are some directions you could explore:

* **Tokenizer algorithm:** BPE, WordPiece and Unigram build vocabularies on
  completely different principles, and they behave differently on Amharic.
* **Pre-tokenization:** how should text be split before vocabulary learning
  begins? This is one of the largest levers available.
* **Language balance:** you have 10,000 slots for six languages but only four
  of them are scored. How should the budget be shared? Remember the guardrail.
* **Normalization:** how should Unicode, casing, punctuation and different
  scripts be handled?
* **Vocabulary design:** how can you make better use of the budget?
* **Combinations:** can several individually useful ideas work together?

**Pick an idea → make one change → measure the result → keep what works.**

# Prepare Your Submission

Found something that beats the baselines?

Export your final tokenizer as `tokenizer.json`, then run the official submission checker.

The checker verifies that your tokenizer:

* Loads correctly with the pinned `tokenizers` version.
* Has no more than **10,000 tokens**.
* Can tokenize text from all six languages.
* Reports the English and French guardrail penalty added to your score.
* Reports your **reconstruction penalty**, charged on any example your
  tokenizer fails to rebuild exactly.
* Meets the required submission format.

It also reports your fertility, your `[UNK]` rate and your final score for each
language, so you can see exactly where the score is coming from before you
submit.

In [ ]:
# Both baselines are scoreable, so keep whichever scores lower.
candidates = [(word_result, word_level), (character_result, character_level)]
best_result, best = min(candidates, key=lambda item: item[0]["score"])

# This is how you save the tokenizer for submission. 
best.save("tokenizer.json", pretty=True)
print(f"Saved {best_result['name']} with score {best_result['score']:.4f}\n")

report = profile_submission("tokenizer.json", data=validation)

## Final Checklist

Before submitting, make sure:

* Your tokenizer has been evaluated on the validation set.
* You have checked its performance across all six languages.
* Your vocabulary contains no more than **10,000 tokens**.
* Your final file is named `tokenizer.json`.
* Your tokenizer passes the submission checker.

Your local validation score is for experimentation.

Your position on the leaderboard is determined using the **hidden test set**.

**Submit your tokenizer, see where you rank, and keep improving.**

# References

This challenge builds on concepts introduced in the **AI Research Foundations** learning path:

* **Course 01: Build Your Own Small Language Model** introduces the language model pipeline and where tokenization fits into building a language model.
* **Course 02: Represent Your Language Data** explores how text is represented for language models, including tokenization and vocabulary construction.

**AI Research Foundations Learning Path:**
https://www.skills.google/paths/3135